In [16]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
import json

In [17]:
# Helper function
def calculate_metrics(y_true, y_pred):
    weighted = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    macro = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

    return {
        "Precision": weighted[0],
        "Recall": weighted[1],
        "F1": weighted[2],
    }, {
        "Precision": macro[0],
        "Recall": macro[1],
        "F1": macro[2],
    }

# Read JSON metrics
with open("../cnsplots/input_data/BacVir95_MLP_averaged_metrics.json") as f:
    metrics_mlp = json.load(f)

with open("../cnsplots/input_data/BacVir95_CNNMLP_averaged_metrics.json") as f:
    metrics_cnnmlp = json.load(f)

with open("../cnsplots/input_data/BacVir95_averaged_metrics.json") as f:
    metrics_95 = json.load(f)

with open("../cnsplots/input_data/BacVir20_averaged_metrics.json") as f:
    metrics_20 = json.load(f)

# Extract BacVir metrics
results = []

results.append({
    "Dataset": "MLP_test",
    "Averaging": "Weighted",
    "Precision": metrics_mlp["bv_test_weighted_precision"],
    "Recall": metrics_mlp["bv_test_weighted_recall"],
    "F1": metrics_mlp["bv_test_weighted_f1"]
})

results.append({
    "Dataset": "MLP_test",
    "Averaging": "Macro",
    "Precision": metrics_mlp["bv_test_macro_precision"],
    "Recall": metrics_mlp["bv_test_macro_recall"],
    "F1": metrics_mlp["bv_test_macro_f1"]
})

results.append({
    "Dataset": "CNN+MLP_test",
    "Averaging": "Weighted",
    "Precision": metrics_cnnmlp["bv_test_weighted_precision"],
    "Recall": metrics_cnnmlp["bv_test_weighted_recall"],
    "F1": metrics_cnnmlp["bv_test_weighted_f1"]
})

results.append({
    "Dataset": "CNN+MLP_test",
    "Averaging": "Macro",
    "Precision": metrics_cnnmlp["bv_test_macro_precision"],
    "Recall": metrics_cnnmlp["bv_test_macro_recall"],
    "F1": metrics_cnnmlp["bv_test_macro_f1"]
})

results.append({
    "Dataset": "PhageGAP95_test(CNN)",
    "Averaging": "Weighted",
    "Precision": metrics_95["bv_test_weighted_precision"],
    "Recall": metrics_95["bv_test_weighted_recall"],
    "F1": metrics_95["bv_test_weighted_f1"]
})

results.append({
    "Dataset": "PhageGAP95_test(CNN)",
    "Averaging": "Macro",
    "Precision": metrics_95["bv_test_macro_precision"],
    "Recall": metrics_95["bv_test_macro_recall"],
    "F1": metrics_95["bv_test_macro_f1"]
})

results.append({
    "Dataset": "PhageGAP20_test",
    "Averaging": "Weighted",
    "Precision": metrics_20["bv_test_weighted_precision"],
    "Recall": metrics_20["bv_test_weighted_recall"],
    "F1": metrics_20["bv_test_weighted_f1"]
})

results.append({
    "Dataset": "PhageGAP20_test",
    "Averaging": "Macro",
    "Precision": metrics_20["bv_test_macro_precision"],
    "Recall": metrics_20["bv_test_macro_recall"],
    "F1": metrics_20["bv_test_macro_f1"]
})

# BASEL 95
basel_95 = pd.read_csv("../cnsplots/input_data/BASEL_BacVir95_final.tsv", sep="\t")

def clean_df(df, colname):
    """"Helper function to clean the dataframe by replacing entries with semicolons in the specified column with 'Unknown'."""
    mask = df[colname].str.contains(";")
    df.loc[mask, colname] = "Unknown"
    return df

basel_95 = clean_df(basel_95, "assigned_subcat_initial")
basel_95_known = basel_95[basel_95["assigned_subcat_initial"] != "Unknown"]

weighted, macro = calculate_metrics(basel_95_known["assigned_subcat_initial"], basel_95_known["Predicted Class"])

results.append({
    "Dataset": "PhageGAP95_BASEL",
    "Averaging": "Weighted",
    **weighted
})

results.append({
    "Dataset": "PhageGAP95_BASEL",
    "Averaging": "Macro",
    **macro
})

# BASEL 20
basel_20 = pd.read_csv("../cnsplots/input_data/BASEL_BacVir20_final.tsv", sep="\t")
basel_20 = clean_df(basel_20, "assigned_subcat_initial")
basel_20_known = basel_20[basel_20["assigned_subcat_initial"] != "Unknown"]

weighted, macro = calculate_metrics(basel_20_known["assigned_subcat_initial"], basel_20_known["Predicted Class"])

results.append({
    "Dataset": "PhageGAP20_BASEL",
    "Averaging": "Weighted",
    **weighted
})

results.append({
    "Dataset": "PhageGAP20_BASEL",
    "Averaging": "Macro",
    **macro
})

# Create table
df_metrics = pd.DataFrame(results)

# Round for reporting
df_metrics[["Precision", "Recall", "F1"]] = (df_metrics[["Precision", "Recall", "F1"]].round(2))

# Save
df_metrics.to_csv("TableS2_metrics.csv", index=False)

print(df_metrics)

                 Dataset Averaging  Precision  Recall    F1
0               MLP_test  Weighted       0.92    0.92  0.92
1               MLP_test     Macro       0.88    0.87  0.87
2           CNN+MLP_test  Weighted       0.93    0.93  0.93
3           CNN+MLP_test     Macro       0.89    0.87  0.87
4   PhageGAP95_test(CNN)  Weighted       0.94    0.93  0.93
5   PhageGAP95_test(CNN)     Macro       0.90    0.88  0.89
6        PhageGAP20_test  Weighted       0.88    0.87  0.87
7        PhageGAP20_test     Macro       0.84    0.81  0.81
8       PhageGAP95_BASEL  Weighted       0.97    0.96  0.96
9       PhageGAP95_BASEL     Macro       0.93    0.93  0.93
10      PhageGAP20_BASEL  Weighted       0.96    0.96  0.96
11      PhageGAP20_BASEL     Macro       0.89    0.91  0.89
